In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS customer")
spark.sql("USE CATALOG customer")

spark.sql("CREATE SCHEMA IF NOT EXISTS transaction")
spark.sql("CREATE SCHEMA IF NOT EXISTS feedback")
spark.sql("CREATE SCHEMA IF NOT EXISTS control")

print("Catalog and schemas created")

display(spark.sql("SHOW SCHEMAS IN customer"))

In [0]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import *

def log_table_access(user_name, table_name, operation):
    audit_entry = spark.createDataFrame([
        Row(
            user_name=user_name,
            table_name=table_name,
            operation=operation,
            timestamp=datetime.now(),
            records_affected=None,
            details=f"{operation} operation on {table_name}"
        )
    ])
    
    audit_entry.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("customer.control.audit_log")
    
    print(f"Logged: {user_name} performed {operation} on {table_name}")

log_table_access("analyst@company.com", "customer.transaction.gold_dim_customer", "SELECT")
log_table_access("analyst@company.com", "customer.transaction.gold_dim_customer_masked", "SELECT")
log_table_access("analyst@company.com", "customer.transaction.gold_fact_sales", "SELECT")
log_table_access("engineer@company.com", "customer.transaction.silver_customers_scd", "SELECT")

print("Access events logged")

In [0]:
def log_data_modification(user_name, table_name, operation, records_affected):
    audit_entry = spark.createDataFrame([
        Row(
            user_name=user_name,
            table_name=table_name,
            operation=operation,
            timestamp=datetime.now(),
            records_affected=records_affected,
            details=f"{operation} {records_affected} records in {table_name}"
        )
    ])
    
    audit_entry.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("customer.control.audit_log")
    
    print(f"Logged: {user_name} {operation} {records_affected} records in {table_name}")

log_data_modification("admin@company.com", "customer.transaction.gold_dim_customer", "UPDATE", 50)
log_data_modification("etl_service", "customer.transaction.bronze_transactions", "INSERT", 5000)
log_data_modification("engineer@company.com", "customer.transaction.silver_customers_scd", "MERGE", 150)

print("Modification events logged")

In [0]:
df_audit = spark.table("customer.control.audit_log")

print(f"Total audit records: {df_audit.count()}")
display(df_audit.orderBy(col("timestamp").desc()).limit(20))

In [0]:
df_user_activity = df_audit.groupBy("user_name", "operation") \
    .agg(count("*").alias("operation_count")) \
    .orderBy(desc("operation_count"))

print("User Activity:")
display(df_user_activity)

df_table_activity = df_audit.groupBy("table_name", "operation") \
    .agg(count("*").alias("operation_count")) \
    .orderBy(desc("operation_count"))

print("Table Activity:")
display(df_table_activity)

In [0]:
print("Access to Sensitive Customer Data:")
df_sensitive_access = spark.sql("""
    SELECT 
        user_name,
        table_name,
        operation,
        timestamp,
        details
    FROM customer.control.audit_log
    WHERE table_name LIKE '%customer%'
    ORDER BY timestamp DESC
""")
display(df_sensitive_access.limit(15))

print("Data Modification Summary:")
df_modifications = spark.sql("""
    SELECT 
        date(timestamp) as activity_date,
        operation,
        COUNT(*) as operation_count,
        SUM(COALESCE(records_affected, 0)) as total_records_affected
    FROM customer.control.audit_log
    WHERE operation IN ('INSERT', 'UPDATE', 'DELETE', 'MERGE')
    GROUP BY date(timestamp), operation
    ORDER BY activity_date DESC, operation_count DESC
""")
display(df_modifications)

In [0]:
df_after_hours = spark.sql("""
    SELECT 
        user_name,
        table_name,
        operation,
        timestamp,
        hour(timestamp) as access_hour
    FROM customer.control.audit_log
    WHERE hour(timestamp) < 8 OR hour(timestamp) > 18
    ORDER BY timestamp DESC
""")

after_hours_count = df_after_hours.count()
print(f"After-hours access: {after_hours_count} events")
if after_hours_count > 0:
    display(df_after_hours.limit(10))

df_high_freq = spark.sql("""
    SELECT 
        user_name,
        COUNT(*) as access_count,
        COUNT(DISTINCT table_name) as distinct_tables
    FROM customer.control.audit_log
    WHERE date(timestamp) = current_date()
    GROUP BY user_name
    HAVING access_count > 10
    ORDER BY access_count DESC
""")

high_freq_count = df_high_freq.count()
print(f"High-frequency access patterns: {high_freq_count} users")
if high_freq_count > 0:
    display(df_high_freq)

In [0]:
df_compliance_report = spark.sql("""
    SELECT 
        log_id,
        user_name,
        table_name,
        operation,
        timestamp,
        date(timestamp) as activity_date,
        hour(timestamp) as activity_hour,
        dayofweek(timestamp) as day_of_week,
        records_affected,
        details,
        CASE 
            WHEN hour(timestamp) BETWEEN 8 AND 18 THEN 'Business Hours'
            ELSE 'After Hours'
        END as access_window,
        CASE 
            WHEN operation IN ('INSERT', 'UPDATE', 'DELETE', 'MERGE') THEN 'Modification'
            WHEN operation = 'SELECT' THEN 'Read Access'
            ELSE 'Other'
        END as activity_type
    FROM customer.control.audit_log
    ORDER BY timestamp DESC
""")

df_compliance_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer.control.audit_compliance_report")

print(f"Compliance report generated: {df_compliance_report.count()} audit entries")
print("Saved to: customer.control.audit_compliance_report")

summary = df_compliance_report.groupBy("activity_type", "access_window") \
    .agg(count("*").alias("event_count")) \
    .orderBy("activity_type", "access_window")

print("Compliance Report Summary:")
display(summary)

from datetime import datetime, timedelta
import random

spark.sql("CREATE VOLUME IF NOT EXISTS customer.transaction.raw_data")
print("Volume created: customer.transaction.raw_data")

def generate_transactions_for_day(date, num_rows=1000):
    data = []
    for i in range(num_rows):
        hour = random.randint(0, 23)
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        
        transaction_time = datetime.combine(date, datetime.min.time()) + timedelta(hours=hour, minutes=minute, seconds=second)
        
        data.append({
            'transaction_id': f'TXN{date.strftime("%Y%m%d")}{i:04d}',
            'transaction_date': date.strftime('%Y-%m-%d'),
            'transaction_timestamp': transaction_time.strftime('%Y-%m-%d %H:%M:%S'),
            'customer_id': random.randint(1, 200),
            'product_id': random.randint(1, 10),
            'store_id': random.randint(1, 5),
            'quantity': random.randint(1, 5),
            'unit_price': round(random.uniform(10.0, 500.0), 2),
            'total_amount': 0,
            'payment_method': random.choice(['credit_card', 'debit_card', 'cash', 'digital_wallet']),
            'status': random.choice(['completed', 'completed', 'completed', 'pending', 'cancelled'])
        })
        data[-1]['total_amount'] = round(data[-1]['quantity'] * data[-1]['unit_price'], 2)
    
    return data

base_date = datetime(2026, 4, 10).date()

for day_offset in range(3):
    current_date = base_date + timedelta(days=day_offset)
    transactions = generate_transactions_for_day(current_date, 1000)
    df = spark.createDataFrame(transactions)
    file_path = f"/Volumes/customer/transaction/raw_data/transactions_{current_date.strftime('%Y%m%d')}.csv"
    
    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(file_path)
    
    print(f"Created: transactions_{current_date.strftime('%Y%m%d')}.csv ({len(transactions)} rows)")

print("3 CSV files created in /Volumes/customer/transaction/raw_data/")